# DOM capture fidelity of FrontMind AI — experiment notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rosas04/Proyecto-Tesis/blob/main/experimento-articulo/FrontMind_DOM_fidelity.ipynb)

Evidence notebook for the article *Structural fidelity of rendered Document Object Model capture: An empirical
evaluation of the FrontMind AI capture agent on 15 web systems* (L.M.A. Rosas Morales, Universidad Privada Antenor Orrego).

*Cuaderno de evidencia del experimento. Ejecutar en Google Colab con **Entorno de ejecución → Ejecutar todas**.*

| Part | Content | Article |
|---|---|---|
| A | Data set of the 15 web systems (A01–A15) | Tables 3–4 |
| B | Descriptive statistics, cumulative subsamples and bootstrap BCa intervals | Table 5 |
| C | Inference on the mean (t-test, BCa and studentized bootstrap), threshold sensitivity, location analysis (Wilcoxon, Hodges–Lehmann) | Section 4.2, Tables 7–8 |
| D | Factors associated with node loss (Spearman, Mann–Whitney) | Section 4.3 |
| E | Reproducibility across runs | Section 4.4, Table 6 |
| F | Figures 2 and 3 | Figs. 2–3 |
| G | Live re-measurement, per-tag classification, protocol equivalence and body-tree correspondence | Sections 3.4, 4.3 |
| H | Comparison with Selenium, Puppeteer and jsdom on A11–A15 | Section 4.6, Table 9 |

Parts A–F only need the recorded counts and run in a few seconds. Part G installs Playwright/Chromium and
re-measures the public systems; set `RUN_BUILDS = True` in G.2 to also rebuild A14 and A15 from their source code.

## Setup

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/Rosas04/Proyecto-Tesis.git"
BRANCH = "main"
if os.path.exists("data/fidelity_by_system.csv"):          # run from inside experimento-articulo/
    EXP = os.path.abspath(".")
else:                                                       # Google Colab: clone the repository
    if not os.path.exists("Proyecto-Tesis"):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL], check=True)
    EXP = os.path.abspath("Proyecto-Tesis/experimento-articulo")
AGENTS = os.path.join(os.path.dirname(EXP), "frontmind-agents")
print("experiment folder:", EXP)
print("frontmind-agents :", AGENTS)

import numpy as np, pandas as pd, scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__,
      "| SciPy", scipy.__version__, "| Matplotlib", matplotlib.__version__)
SEED = 2026
ALPHA, CRITERION = 0.05, 90.0

## A. Data set (Tables 3–4)

Original nodes: element nodes of the rendered DOM before sanitization; captured nodes: element nodes of the replica after sanitization; both summed over the routes visited. Counts were identical in the three runs of every system.

In [ ]:
df = pd.read_csv(os.path.join(EXP, "data", "fidelity_by_system.csv"))
df["removed_nodes"] = df.original_nodes - df.captured_nodes
df["fidelity"] = df.captured_nodes / df.original_nodes * 100                   # Eq. (1)
df["nodes_per_route"] = df.original_nodes / df.routes
df["removed_per_route"] = df.removed_nodes / df.routes
t3 = df[["id", "stage", "system", "routes", "original_nodes", "captured_nodes", "removed_nodes", "fidelity"]].copy()
t3["fidelity"] = t3.fidelity.round(2)
f_agg = df.captured_nodes.sum() / df.original_nodes.sum() * 100                # Eq. (2)
print(t3.to_string(index=False))
print(f"\nTotal: {df.original_nodes.sum():,} original, {df.captured_nodes.sum():,} captured, "
      f"{df.removed_nodes.sum()} removed -> aggregate fidelity F_agg = {f_agg:.2f}%")
print(f"Unweighted mean of the 15 systems = {df.fidelity.mean():.2f}%")

## B. Descriptive statistics per cumulative subsample (Table 5)

Besides the parametric t interval of the mean, the 95% bias-corrected and accelerated (BCa) bootstrap intervals of the mean and of the median are computed (10,000 resamples, fixed seed).

In [ ]:
def bca(x, fn):
    r = stats.bootstrap((x,), fn, confidence_level=0.95, n_resamples=10_000, method="BCa",
                        random_state=np.random.default_rng(SEED))
    return r.confidence_interval.low, r.confidence_interval.high

def describe(x, o, c):
    n = len(x); m = x.mean(); sd = x.std(ddof=1); se = sd / np.sqrt(n); tc = stats.t.ppf(0.975, n - 1)
    sw = stats.shapiro(np.round(x, 2))
    return {"n": n, "Mean (%)": m, "Median (%)": np.median(x), "SD": sd, "SE": se, "CV (%)": sd / m * 100,
            "Min (%)": x.min(), "Max (%)": x.max(), "Range": x.max() - x.min(),
            "95% CI mean (t)": (m - tc * se, m + tc * se), "95% CI mean (BCa)": bca(x, np.mean),
            "95% CI median (BCa)": bca(x, np.median),
            "Original/captured": f"{o.sum():,} / {c.sum():,}", "Aggregate (%)": c.sum() / o.sum() * 100,
            "Systems > 90%": f"{(x > CRITERION).sum()} of {n}", "Shapiro-Wilk W (p)": (sw.statistic, sw.pvalue)}

subsamples = {"Stage 1 (n = 5)": df.index < 5, "Stages 1-2 (n = 10)": df.index < 10,
              "Stages 1-3 (n = 13)": df.index < 13, "Full sample (n = 15)": df.index < 15}
T4 = {k: describe(df.fidelity[m].to_numpy(), df.original_nodes[m], df.captured_nodes[m]) for k, m in subsamples.items()}
fmt = lambda v: (f"[{v[0]:.2f}, {v[1]:.2f}]" if isinstance(v, tuple) and len(v) == 2 and v[0] > 1
                 else f"{v[0]:.3f} ({v[1]:.4f})" if isinstance(v, tuple) else f"{v:.2f}" if isinstance(v, float) else v)
print(pd.DataFrame({k: {s: fmt(v) for s, v in d.items()} for k, d in T4.items()}).to_string())

## C. Hypothesis testing (Section 4.2 and Table 7)

H0: μ ≤ 90% vs. H1: μ > 90%, one-sided, α = 0.05. The hypothesis concerns the **mean**, so it is tested with the one-sample t-test and its one-sided 95% lower bound. Because Shapiro–Wilk rejects normality, the t-test is checked with two bootstrap procedures that do not assume it (one-sided BCa lower bound and studentized bootstrap-t test; 10,000 resamples, seed 2026). The Wilcoxon test concerns the center of the distribution, not the mean, and is reported as a complementary location analysis (C.2). 'Rejected (marginal)': the t-test rejects H0 and the studentized bootstrap does not.

In [ ]:
def hedges_g(x):
    xr = np.round(x, 2); n = len(xr)
    return (xr.mean() - CRITERION) / xr.std(ddof=1) * (1 - 3 / (4 * (n - 1) - 1))

def boot_t(x, mu0=CRITERION, B=10_000, seed=2026):
    rng = np.random.default_rng(seed); n = len(x); m = x.mean(); se = x.std(ddof=1) / np.sqrt(n)
    xb = x[rng.integers(0, n, (B, n))]; mb = xb.mean(1); sb = xb.std(1, ddof=1) / np.sqrt(n)
    ok = sb > 0; tb = (mb[ok] - m) / sb[ok]
    return m - np.quantile(tb, 0.95) * se, (np.sum(tb >= (m - mu0) / se) + 1) / (ok.sum() + 1)

def bca_lower(x):
    r = stats.bootstrap((x,), np.mean, confidence_level=0.90, n_resamples=10_000, method="BCa",
                        random_state=np.random.default_rng(2026))
    return r.confidence_interval.low          # = one-sided 95% lower bound

def test(x):
    n = len(x); se = x.std(ddof=1) / np.sqrt(n)
    t = stats.ttest_1samp(x, CRITERION, alternative="greater")
    w = stats.wilcoxon(x - CRITERION, alternative="greater", method="exact")
    blb, bp = boot_t(x)
    return {"n": n, "mean": x.mean(), "t": t.statistic, "p_t": t.pvalue,
            "t_lower": x.mean() - stats.t.ppf(.95, n - 1) * se, "BCa_lower": bca_lower(x),
            "bootT_lower": blb, "p_bootT": bp, "W": w.statistic, "p_wilcoxon": w.pvalue, "g": hedges_g(x)}

rows = {k: test(df.fidelity[m].to_numpy()) for k, m in subsamples.items()}
rows["Sensitivity: DOM >= 150 nodes"] = test(df.fidelity[df.original_nodes >= 150].to_numpy())
rows["Sensitivity: without A01 and A15"] = test(df.fidelity[~df.id.isin(["A01", "A15"])].to_numpy())
T7 = pd.DataFrame(rows).T
T7["decision"] = np.where(T7.p_t < ALPHA, np.where(T7.p_bootT < ALPHA, "Rejected", "Rejected (marginal)"), "Not rejected")
print(T7.round(4).to_string())
full = T4["Full sample (n = 15)"]
print(f"\nFull sample: Shapiro-Wilk W = {full['Shapiro-Wilk W (p)'][0]:.3f}, p = {full['Shapiro-Wilk W (p)'][1]:.4f}")
print("Two-sided 95% CI of the mean, t (descriptive)  :", fmt(full["95% CI mean (t)"]))
print("Two-sided 95% CI of the mean, BCa (descriptive):", fmt(full["95% CI mean (BCa)"]))

### C.1 Sensitivity of the decision to the threshold (Table 8)

In [ ]:
x = df.fidelity.to_numpy(); rows = []
for th in (85, 87.5, 90, 91, 92.5, 95):
    tp = stats.ttest_1samp(x, th, alternative="greater").pvalue; bp = boot_t(x, th)[1]
    wp = stats.wilcoxon(x - th, alternative="greater").pvalue
    rows.append({"threshold": th, "p_t": tp, "p_bootT": bp, "p_wilcoxon": wp,
                 "systems >= threshold": int((x >= th).sum()),
                 "decision": ("Rejected" if bp < ALPHA else "Rejected (marginal)") if tp < ALPHA else "Not rejected"})
print(pd.DataFrame(rows).round(4).to_string(index=False))

### C.2 Location analysis: interval consistent with the Wilcoxon test (Hodges–Lehmann)

This concerns the center of the distribution (pseudo-median), not the mean. The interval that corresponds to the Wilcoxon signed-rank test is obtained by inverting it: the Hodges–Lehmann estimator is the median of the Walsh averages, and the one-sided 95% lower confidence bound is the (c + 1)-th smallest Walsh average, where c is the largest value with P(W+ ≤ c) ≤ 0.05 under H0 (exact null distribution).

In [ ]:
def hodges_lehmann(x, alpha=0.05):
    n = len(x)
    walsh = np.sort([(x[i] + x[j]) / 2 for i in range(n) for j in range(i, n)])
    dist = np.zeros(n * (n + 1) // 2 + 1); dist[0] = 1
    for k in range(1, n + 1):                 # exact null distribution of W+
        nxt = dist.copy(); nxt[k:] += dist[:-k]; dist = nxt
    cdf = np.cumsum(dist / 2 ** n)
    c = max(i for i in range(len(cdf)) if cdf[i] <= alpha)
    return np.median(walsh), walsh[c], c
for k, m in list(subsamples.items()) + [("Sensitivity: DOM >= 150 nodes", (df.original_nodes >= 150).to_numpy()),
                                       ("Sensitivity: without A01 and A15", (~df.id.isin(["A01", "A15"])).to_numpy())]:
    hl, lb, c = hodges_lehmann(df.fidelity[m].to_numpy())
    print(f"{k:32s} HL = {hl:6.2f}%  one-sided 95% lower bound = {lb:6.2f}%  (c = {c})")

## D. Factors associated with node loss (Section 4.3)

In [ ]:
def rho(a, b):
    r = stats.spearmanr(a, b); return f"rho = {r.statistic:.2f}, p = {r.pvalue:.4f}"
print("System size vs fidelity          :", rho(df.original_nodes, df.fidelity))
print("System size vs removed nodes     :", rho(df.original_nodes, df.removed_nodes))
print("Nodes per route vs fidelity      :", rho(df.nodes_per_route, df.fidelity))
print("Removed per route vs fidelity    :", rho(df.removed_per_route, df.fidelity))
small = df.original_nodes < 150
u = stats.mannwhitneyu(df.fidelity[small], df.fidelity[~small], alternative="less", method="exact")
print(f"Mann-Whitney (<150 vs >=150 nodes): U = {u.statistic:.0f}, p = {u.pvalue:.5f}")
print(df.loc[df.fidelity < CRITERION, ["id", "original_nodes", "routes", "nodes_per_route",
                                       "removed_per_route", "fidelity"]].round(2).to_string(index=False))

## E. Reproducibility across runs (Table 6)

In [ ]:
times = df[["time_run1_s", "time_run2_s", "time_run3_s"]]
t5 = pd.DataFrame({"id": df.id, "run 1": times.iloc[:, 0], "run 2": times.iloc[:, 1], "run 3": times.iloc[:, 2],
                   "mean (s)": times.mean(axis=1).round(2), "SD (s)": times.std(axis=1, ddof=1).round(2)})
print(t5.to_string(index=False))
print("\nCapture time vs DOM size:", rho(df.original_nodes, times.mean(axis=1)))
print("Run-to-run variance of node counts: 0 in all 45 runs (identical counts, see data/*.json)")

## F. Figures 2 and 3

In [ ]:
BLUE, RED, INK = "#1a6fb0", "#b03a2e", "#333333"
fid = df.fidelity.to_numpy(); ids = df.id.tolist()
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(ids, fid, color=[RED if v < CRITERION else BLUE for v in fid])
for i, v in enumerate(fid):
    ax.text(i, v + 0.3, f"{v:.2f}", ha="center", fontsize=7)
ax.axhline(CRITERION, ls="--", color=INK, lw=0.9, label="criterion 90%")
ax.axhline(fid.mean(), ls=":", color=INK, lw=1.1, label=f"mean {fid.mean():.2f}%")
ax.set_ylim(75, 103); ax.set_ylabel("DOM capture fidelity (%)"); ax.legend(frameon=False, loc="lower right")
ax.set_title("Fig. 2. DOM capture fidelity per web system"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df.nodes_per_route, fid, c=[RED if v < CRITERION else BLUE for v in fid], s=50)
for _, r in df.iterrows():
    ax.annotate(f"{r.id} ({r.removed_per_route:.1f})", (r.nodes_per_route, r.fidelity), xytext=(5, -3),
                textcoords="offset points", fontsize=7)
ax.set_xscale("log"); ax.axhline(CRITERION, ls="--", color=INK, lw=0.9)
ax.set_xlabel("Mean original nodes per route (log scale)"); ax.set_ylabel("DOM capture fidelity (%)")
ax.set_title("Fig. 3. Page size vs. capture fidelity (removed nodes per route)"); plt.tight_layout(); plt.show()

## G. Live re-measurement with the capture procedure

The script `measure_fidelity.py` reproduces the capture agent of FrontMind AI (headless Chromium, 1366 × 768
viewport, network idleness, full scrolling) and its replication agent (`HtmlReplicationAgent` of
`frontmind-agents`), and records for every route the element count per tag name in the original page and in the
replica, so the removed nodes are classified by tag automatically. Deployed systems may have changed since the
measurement dates of the article, so this part checks the procedure; the counts of the article are those of Part A.

In [ ]:
!pip -q install playwright==1.60.0 beautifulsoup4 requests httpx fastapi lxml
!python -m playwright install --with-deps chromium > /dev/null 2>&1 && echo "Chromium ready"

### G.1 Public deployments (A11–A13), three runs

In [ ]:
A11 = "https://sentryx-surveillance.vercel.app/"
A12 = "https://resthub-frontend-zeta.vercel.app/acceso"
A13 = ",".join("https://check-scan-tesis.vercel.app/" + r for r in ["login", "olvide", "privacidad", "condiciones"])
LIVE = os.path.join(EXP, "results", "live_A11_A13_colab.json"); os.makedirs(os.path.dirname(LIVE), exist_ok=True)
!python "{EXP}/measure_fidelity.py" "{AGENTS}" "{LIVE}" 3 A11={A11} A12={A12} A13={A13}

In [ ]:
import json
def summarize(path):
    data = json.load(open(path, encoding="utf-8"))
    out = []
    for sid, runs in data.items():
        removed = {}
        for route in runs[0]["routes"]:
            for tag, k in route["removed_by_tag"].items():
                removed[tag] = removed.get(tag, 0) + k
        out.append({"id": sid, "original (runs)": [r["original"] for r in runs],
                    "captured (runs)": [r["captured"] for r in runs], "fidelity (%)": runs[0]["fidelity"],
                    "identical runs": len({(r["original"], r["captured"]) for r in runs}) == 1,
                    "removed by tag": removed})
    return pd.DataFrame(out)
live = summarize(LIVE)
live["article (%)"] = live.id.map(dict(zip(df.id, df.fidelity.round(2))))
print(live.to_string(index=False))

### G.2 A14 and A15 rebuilt from source (optional, about 10 minutes)

Both repositories are public. They are checked out at the commits measured in the article, built with the production configuration of the Angular CLI and served locally with single-page-application fallback.

In [ ]:
RUN_BUILDS = False   # set to True to rebuild and re-measure A14 and A15
SOURCES = {"A14": ("https://github.com/Cronos7898/inso-2-front-end.git", "70eba48", "dist/fronted-i-love-travel-app", 8714, [""]),
           "A15": ("https://github.com/OptiLifes/RepositorioFront.git", "bd315ea", "dist/opti-life/browser", 8716,
                   ["", "login", "register", "forgot"])}
if RUN_BUILDS:
    import time
    !node -v || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get install -y nodejs > /dev/null)
    args, servers = [], []
    for sid, (url, commit, dist, port, routes) in SOURCES.items():
        src = f"/content/{sid}"
        if not os.path.exists(src):
            subprocess.run(["git", "clone", "-q", url, src], check=True)
        subprocess.run(["git", "-C", src, "checkout", "-q", commit], check=True)
        subprocess.run("npm ci --no-audit --no-fund --loglevel=error && npx ng build", shell=True, cwd=src, check=True)
        servers.append(subprocess.Popen([sys.executable, f"{EXP}/spa_server.py", f"{src}/{dist}", str(port)]))
        args.append(f"{sid}=" + ",".join(f"http://127.0.0.1:{port}/{r}" for r in routes))
    time.sleep(2)
    BUILT = os.path.join(EXP, "results", "live_A14_A15_colab.json")
    subprocess.run([sys.executable, f"{EXP}/measure_fidelity.py", AGENTS, BUILT, "3", *args], check=True)
    for s in servers:
        s.terminate()
    built = summarize(BUILT); built["article (%)"] = built.id.map(dict(zip(df.id, df.fidelity.round(2))))
    print(built.to_string(index=False))
else:
    print("Skipped. Records of A14 and A15 measured on 23 November 2026 (local builds):")
    print(summarize(os.path.join(EXP, "data", "live_A14_A15_23nov2026.json")).to_string(index=False))

### G.3 Automatic classification of the removed nodes (A11–A15)

Difference between the element count per tag name of the original page and of the replica, summed over routes (first run; the three runs are identical). Positive values are elements removed by sanitization; negative values are elements added by the replication agent (inlined `<style>` blocks).

In [ ]:
VISIBLE_SAFE = {"script", "noscript", "link", "meta", "base", "style", "title"}
recs = {}
for f in ["live_A11_A13_23nov2026.json", "live_A14_A15_23nov2026.json"]:
    recs.update(json.load(open(os.path.join(EXP, "data", f), encoding="utf-8")))
rows = []
for sid, runs in recs.items():
    tot = {}
    for route in runs[0]["routes"]:
        for tag, k in route["removed_by_tag"].items():
            tot[tag] = tot.get(tag, 0) + k
    rows.append({"id": sid, "net removed": runs[0]["original"] - runs[0]["captured"], **tot,
                 "only non-rendering tags": set(tot) <= VISIBLE_SAFE})
print(pd.DataFrame(rows).fillna(0).to_string(index=False))

### G.4 Protocol equivalence and body-tree correspondence (A11–A15)

`verify_protocol.py` runs the capture worker of the framework itself (its own route discovery, waits, container wait, tracker blocking and three viewports) and the measurement procedure on the same routes, and compares the body of each rendered page with that of its replica: the sequence of (depth, tag) of every body element except scripts and noscript blocks (identical only if every parent–child relation and sibling order is preserved) and the visible text. The recorded result of 24 November 2026 is shown; set `RUN_PROTOCOL = True` to repeat it on A11–A13.

In [ ]:
RUN_PROTOCOL = False
if RUN_PROTOCOL:
    PR = os.path.join(EXP, "results", "protocol_colab.json")
    A13_START = A13.split(",")[0]
    !python "{EXP}/verify_protocol.py" "{AGENTS}" "{PR}" "A11={A11}|{A11}" "A12={A12}|{A12}" "A13={A13_START}|{A13}"
else:
    PR = os.path.join(EXP, "data", "protocol_A11_A15_24nov2026.json")
pr = json.load(open(PR, encoding="utf-8")); rows = []
for sid, d in pr.items():
    for m, mm in zip(d["measurement"], d["matched"]):
        rows.append({"id": sid, "route": m["url"].split("/", 3)[-1] or "/",
                     "procedure (orig, replica)": tuple(mm["measurement"]),
                     "framework worker (orig, replica)": tuple(mm["framework"]) if mm["framework"] else None,
                     "removed": sum(m["removed_by_tag"].values()), "added": sum(m["added_by_tag"].values()),
                     "body elements": m["body_elements"], "body tree identical": m["body_tree_identical"],
                     "body text identical": m["body_text_identical"]})
print(pd.DataFrame(rows).to_string(index=False))

## H. Comparison with other capture tools (A11–A15)

The same routes were captured with Playwright (reference, the capture agent of FrontMind AI), Selenium WebDriver with a
network-idle-equivalent wait (`selenium`) and with its default load-event wait (`selenium_load`), Puppeteer (all
three driving the same Chromium binary), jsdom executing the page scripts, and jsdom without scripts (the
HTML served before JavaScript runs). *native* is the element count in each tool's own DOM; *replica* is the count of
the replica that `HtmlReplicationAgent` builds from that tool's HTML, rendered in Chromium. Fidelity of each tool is
its replica count divided by the rendered DOM obtained by Playwright.

In [ ]:
RUN_COMPARISON = False   # True: re-run the comparison on the public systems A11-A13 (about 10 minutes)
if RUN_COMPARISON:
    !pip -q install selenium==4.54.0
    !cd "{EXP}/tools" && npm install --no-audit --no-fund --loglevel=error
    CMP = os.path.join(EXP, "results", "tools_comparison_colab.json")
    !python "{EXP}/compare_tools.py" "{AGENTS}" "{EXP}/tools" "{CMP}" 3 A11={A11} A12={A12} A13={A13}
else:
    CMP = os.path.join(EXP, "data", "tools_comparison_A11_A15_24nov2026.json")
cmp = json.load(open(CMP, encoding="utf-8"))
rows = []
for sid, tools in cmp.items():
    ref = tools["playwright"][0]["native"]
    for tool, runs in tools.items():
        rows.append({"id": sid, "tool": tool, "native (runs)": [r["native"] for r in runs],
                     "replica (runs)": [r["replica"] for r in runs],
                     "acquired (% of rendered DOM)": round(runs[0]["native"] / ref * 100, 2),
                     "fidelity (%)": round(runs[0]["replica"] / ref * 100, 2),
                     "identical runs": len({(r["native"], r["replica"]) for r in runs}) == 1})
cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))
print()
print(cmp_df.pivot(index="id", columns="tool", values="fidelity (%)").to_string())

## Records included in `data/`

* `fidelity_by_system.csv`: per-system counts and capture times of the 15 systems (Tables 2, 3 and 5).
* `runs_A14_A15_23nov2026.json`: per-run and per-route records of A14 and A15 measured on 23 November 2026.
* `live_A11_A13_23nov2026.json`: per-run, per-route and per-tag records of the re-measurement of the deployed A11–A13.
* `live_A14_A15_23nov2026.json`: per-run, per-route and per-tag records of A14 and A15 (local production builds).
* `tools_comparison_A11_A15_24nov2026.json`: comparison of Playwright, Selenium, Puppeteer and jsdom on A11–A15 (Part H).
* `protocol_A11_A15_24nov2026.json`: framework capture worker vs. measurement procedure and body-tree correspondence (Part G.4).

A01–A10 were measured inside the FrontMind AI framework (run logs); their counts are reported as recorded.